# Random Forest and XGBoost on Morgan Fingerprints (Representation B)

This notebook uses Morgan fingerprints (refer to notebook 1b) and compares Random Forest and XGBoost models using k-fold cross-validation.

**Goal:** Compare Random Forest and XGBoost performance on fingerprint representation through k-fold cross-validation to determine which model performs best.



In [ ]:
### Import Required Libraries

- **numpy**: Data manipulation and array operations
- **scipy.stats**: Statistical tests and confidence intervals
- **sklearn**: Machine learning tools (models, cross-validation, metrics)
- **xgboost**: Gradient boosting model (XGBClassifier)



In [ ]:
import numpy as np
from scipy import stats

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import RandomForestClassifier

# XGBoost
try:
    from xgboost import XGBClassifier
except ImportError:
    raise ImportError(
        "xgboost is not installed in this environment. Install it with `pip install xgboost` or `conda install -c conda-forge xgboost`."
    )

#ignore warnings
import warnings
warnings.filterwarnings('ignore')



In [ ]:
## Step 1: Load Fingerprint Data

Load the fingerprint matrix and labels generated in the previous notebook (`1_get_fingerprints.ipynb`).

This dataset contains Morgan fingerprints (2048-bit vectors) plus the target variable (ACTIVE: 0 = inactive, 1 = active).



In [ ]:
X_fp = np.load("training_fingerprints_matrix.npy")
y_fp = np.load("training_fingerprints_labels.npy")

print(f"Fingerprint matrix shape: {X_fp.shape}")
print(f"Labels shape: {y_fp.shape}")
print(f"Class balance: {(y_fp==1).sum()} active / {(y_fp==0).sum()} inactive")



In [ ]:
## Step 2: Define Helper Function and Cross-Validation

Set random seed for reproducibility and define a helper function to calculate confidence intervals. This ensures that results are consistent across runs.



In [ ]:
seed = 20231124
np.random.seed(seed)

def calculate_ci(scores, confidence=0.95):
    n = len(scores)
    mean = np.mean(scores)
    std_err = stats.sem(scores)  # Standard error of the mean
    ci = stats.t.interval(confidence, n-1, loc=mean, scale=std_err)
    return mean, ci



In [ ]:
## Step 3: Train Machine Learning Models

Train both **Random Forest** and **XGBoost** classifiers to predict molecular activity using fingerprints.

**Process:**

1. **10-Fold Cross-Validation**: 
   - Split data into 10 folds - use StratifiedKFold to distribute classes better
   - Train on 9 folds, test on 1 fold
   - Repeat 10 times with different splits
   - This gives a robust estimate of model performance

2. **XGBoost Classifier**
   - Gradient boosting model optimized for tabular data
   - Uses hyperparameters from previous tuning experiments

3. **Random Forest Classifier**:
   - Ensemble method that combines multiple decision trees
   - Good baseline model for structured data
   - Handles many features well

**Output:** Mean ROC AUC across all 10 folds with confidence intervals for both models



## 3a - XGBoost evaluation on Kfolds


In [ ]:
cv = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=seed
)

xgb_clf = XGBClassifier(
    n_estimators=600,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.9,
    colsample_bytree=0.7,
    reg_alpha=0.3,
    reg_lambda=1.0,
    min_child_weight=1,
    gamma=0.0,
    objective="binary:logistic",
    eval_metric="auc",
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

xgb_auc = cross_val_score(
    xgb_clf,
    X_fp,
    y_fp,
    cv=cv,
    scoring="roc_auc",
    n_jobs=1
)

xgb_mean, xgb_ci = calculate_ci(xgb_auc)

print("AUC per fold:", xgb_auc)
print("Mean AUC:", xgb_mean)
print("Std AUC:", xgb_auc.std())
print(f"95% CI: [{xgb_ci[0]:.4f}, {xgb_ci[1]:.4f}]")


## 3b - Random Forest evaluation on kfolds


In [ ]:
rf_clf = RandomForestClassifier(
    n_estimators=300,
    max_features="sqrt",
    min_samples_leaf=2,
    n_jobs=-1,
    random_state=42
)

rf_auc = cross_val_score(
    rf_clf,
    X_fp,
    y_fp,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1
)

rf_mean, rf_ci = calculate_ci(rf_auc)

print("AUC per fold:", rf_auc)
print("Mean AUC:", rf_mean)
print("Std AUC:", rf_auc.std())
print(f"95% CI: [{rf_ci[0]:.4f}, {rf_ci[1]:.4f}]")


## 3c - Comparison and statistical test

Compare the performance of XGBoost and Random Forest on fingerprints using statistical tests to determine if the difference is significant.


In [ ]:
# display comparison
print("XGBoost results: ")
print("AUC per fold:", np.round(xgb_auc, 4))
print(f"Mean AUC: {xgb_mean:.4f}")
print(f"Std AUC: {xgb_auc.std():.4f}")
print(f"95% CI: [{xgb_ci[0]:.4f}, {xgb_ci[1]:.4f}]")
print(f"CI difference: {xgb_ci[1]-xgb_ci[0]:.4f}")
print("*******")
print("Random Forest results: ")
print("AUC per fold:", np.round(rf_auc, 4))
print(f"Mean AUC: {rf_mean:.4f}")
print(f"Std AUC: {rf_auc.std():.4f}")
print(f"95% CI: [{rf_ci[0]:.4f}, {rf_ci[1]:.4f}]")
print(f"CI difference: {rf_ci[1]-rf_ci[0]:.4f}")


In [ ]:
# because of the overlap, use ttest to check if the results from the models performance mean it could be
# statistically significant
t_stat, p_value = stats.ttest_rel(xgb_auc, rf_auc)
print(f"Difference in means: {abs(xgb_mean - rf_mean):.4f}")
print(f"Paired t-test p-value: {p_value:.4f}")
if p_value < 0.05:
    winner = "XGBoost" if xgb_mean > rf_mean else "Random Forest"
    print(f"Winner: {winner} (statistically significant)")
else:
    print("No statistically significant difference between models")
